In [ ]:
import torch
import torch.nn as nn

# ══════════════════════════════════════════════
#  1. VANILLA RNN CELL FROM SCRATCH
# ══════════════════════════════════════════════

class RNNCellScratch(nn.Module):
    """
    Vanilla RNN Cell — processes ONE timestep.

    The core equation:
        h_t = tanh(W_ih @ x_t + b_ih + W_hh @ h_{t-1} + b_hh)

    Inputs:
        x_t:   current input        (batch, input_size)
        h_prev: previous hidden state (batch, hidden_size)

    Output:
        h_t:   new hidden state      (batch, hidden_size)
    """

    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.input_size  = input_size
        self.hidden_size = hidden_size

        # Learnable parameters
        self.W_ih = nn.Parameter(torch.randn(hidden_size, input_size)  * 0.01)  # input → hidden
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)  # hidden → hidden
        self.b_ih = nn.Parameter(torch.zeros(hidden_size))
        self.b_hh = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, x_t: torch.Tensor,
                h_prev: torch.Tensor = None) -> torch.Tensor:
        """
        One timestep:
            h_t = tanh( x_t @ W_ih^T + b_ih  +  h_{t-1} @ W_hh^T + b_hh )
        """
        if h_prev is None:
            h_prev = torch.zeros(x_t.size(0), self.hidden_size, device=x_t.device)

        h_t = torch.tanh(
            x_t   @ self.W_ih.t() + self.b_ih +
            h_prev @ self.W_hh.t() + self.b_hh
        )
        return h_t


# ══════════════════════════════════════════════
#  2. VANILLA RNN (processes full sequences)
# ══════════════════════════════════════════════

class RNNScratch(nn.Module):
    """
    Full RNN — unrolls an RNN cell across all timesteps.

    Input:  (batch, seq_len, input_size)
    Output: all hidden states + final hidden state
    """

    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.cell = RNNCellScratch(input_size, hidden_size)
        self.hidden_size = hidden_size

    def forward(self, x: torch.Tensor,
                h_0: torch.Tensor = None) -> tuple:
        """
        Unroll through time:
            for t in range(seq_len):
                h_t = cell(x[:, t, :], h_{t-1})

        Returns:
            output: all hidden states  (batch, seq_len, hidden_size)
            h_n:    final hidden state  (batch, hidden_size)
        """
        B, T, _ = x.shape

        if h_0 is None:
            h_0 = torch.zeros(B, self.hidden_size, device=x.device)

        h_t = h_0
        outputs = []

        for t in range(T):
            h_t = self.cell(x[:, t, :], h_t)
            outputs.append(h_t)

        # Stack: list of (B, H) → (B, T, H)
        output = torch.stack(outputs, dim=1)
        return output, h_t


# ══════════════════════════════════════════════
#  3. STEP-BY-STEP WALKTHROUGH
# ══════════════════════════════════════════════

def step_by_step():
    """Trace every value through an RNN cell, one timestep at a time."""
    print("── Step-by-step: 3 timesteps ──\n")

    torch.manual_seed(42)

    cell = RNNCellScratch(input_size=2, hidden_size=3)

    # A short sequence: 3 timesteps, 2 features each
    sequence = torch.tensor([
        [1.0, 0.5],    # t=0
        [0.3, 0.8],    # t=1
        [0.7, 0.2],    # t=2
    ])

    h = torch.zeros(1, 3)   # initial hidden state

    for t in range(len(sequence)):
        x_t = sequence[t].unsqueeze(0)    # (1, 2)
        h = cell(x_t, h)

        print(f"  t={t}  input: {sequence[t].tolist()}")
        print(f"       hidden: {[f'{v:.4f}' for v in h.squeeze().tolist()]}")

        # Show the math for t=0
        if t == 0:
            raw = x_t @ cell.W_ih.t() + cell.b_ih + torch.zeros(1, 3) @ cell.W_hh.t() + cell.b_hh
            print(f"       pre-tanh: {[f'{v:.4f}' for v in raw.squeeze().tolist()]}")
            print(f"       → tanh squashes to [-1, 1]")
        print()


# ══════════════════════════════════════════════
#  4. VERIFY AGAINST PYTORCH
# ══════════════════════════════════════════════

def verify():
    """Prove our RNN matches PyTorch's nn.RNN exactly."""
    print("── Verify against PyTorch ──\n")

    torch.manual_seed(7)

    input_size  = 4
    hidden_size = 5
    seq_len     = 6
    batch_size  = 2

    x = torch.randn(batch_size, seq_len, input_size)

    # ── Our RNN ──
    rnn_ours = RNNScratch(input_size, hidden_size)

    # ── PyTorch RNN ──
    rnn_pt = nn.RNN(input_size, hidden_size, batch_first=True)

    # Copy weights: PyTorch packs W_ih and W_hh into single tensors
    rnn_ours.cell.W_ih.data = rnn_pt.weight_ih_l0.data.clone()
    rnn_ours.cell.W_hh.data = rnn_pt.weight_hh_l0.data.clone()
    rnn_ours.cell.b_ih.data = rnn_pt.bias_ih_l0.data.clone()
    rnn_ours.cell.b_hh.data = rnn_pt.bias_hh_l0.data.clone()

    # Forward
    out_ours, h_ours = rnn_ours(x)
    out_pt,   h_pt   = rnn_pt(x)
    h_pt = h_pt.squeeze(0)   # PT returns (1, B, H) → (B, H)

    out_match = torch.allclose(out_ours, out_pt, atol=1e-6)
    h_match   = torch.allclose(h_ours, h_pt, atol=1e-6)

    print(f"  Input:          {list(x.shape)}")
    print(f"  Our output:     {list(out_ours.shape)}")
    print(f"  PT  output:     {list(out_pt.shape)}")
    print(f"  Output match:   {out_match} {'✓' if out_match else '✗'}")
    print(f"  Hidden match:   {h_match} {'✓' if h_match else '✗'}")
    print()


# ══════════════════════════════════════════════
#  5. SEQUENCE CLASSIFICATION EXAMPLE
# ══════════════════════════════════════════════

def sequence_classification():
    """Train our RNN on a toy problem: is the sum of the sequence > 0?"""
    print("── Train: sequence classification ──\n")

    torch.manual_seed(42)

    # Generate sequences: (batch, seq_len=8, features=1)
    n_samples = 200
    seq_len   = 8
    X = torch.randn(n_samples, seq_len, 1)
    y = (X.sum(dim=(1, 2)) > 0).float().unsqueeze(1)   # 1 if sum > 0

    # Split
    X_train, X_val = X[:150], X[150:]
    y_train, y_val = y[:150], y[150:]

    # Model: RNN → take last hidden → linear → sigmoid
    class SeqClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            self.rnn = RNNScratch(input_size=1, hidden_size=16)
            self.fc  = nn.Linear(16, 1)

        def forward(self, x):
            _, h_last = self.rnn(x)        # take final hidden state
            return torch.sigmoid(self.fc(h_last))

    model = SeqClassifier()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.BCELoss()

    print(f"  Task: is sum(sequence) > 0?")
    print(f"  Data: {X_train.shape[0]} train, {X_val.shape[0]} val, seq_len={seq_len}\n")

    for epoch in range(1, 101):
        # Train
        model.train()
        pred = model(X_train)
        loss = criterion(pred, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 25 == 0:
            # Eval
            model.eval()
            with torch.no_grad():
                val_pred = model(X_val)
                val_loss = criterion(val_pred, y_val)
                val_acc  = ((val_pred >= 0.5).float() == y_val).float().mean()

            print(f"  Epoch {epoch:3d} | Loss: {loss.item():.4f} | "
                  f"Val Loss: {val_loss.item():.4f} | Val Acc: {val_acc:.0%}")

    # Show some predictions
    model.eval()
    print(f"\n  Sample predictions:")
    with torch.no_grad():
        for i in range(5):
            seq = X_val[i]
            true = int(y_val[i].item())
            prob = model(seq.unsqueeze(0)).item()
            pred_label = int(prob >= 0.5)
            status = "✓" if pred_label == true else "✗"
            print(f"    sum={seq.sum().item():+.2f}  target={true}  "
                  f"pred={prob:.3f}→{pred_label}  {status}")
    print()


# ══════════════════════════════════════════════
#  6. CHARACTER-LEVEL DEMO (next char prediction)
# ══════════════════════════════════════════════

def char_demo():
    """Show how RNN maintains memory over a character sequence."""
    print("── Demo: character-level hidden state evolution ──\n")

    torch.manual_seed(0)

    chars = "hello world"
    vocab = sorted(set(chars))
    c2i = {c: i for i, c in enumerate(vocab)}

    cell = RNNCellScratch(input_size=len(vocab), hidden_size=8)
    h = torch.zeros(1, 8)

    print(f"  Sequence: '{chars}'")
    print(f"  Vocab:    {vocab}")
    print(f"  Hidden:   8 dims\n")
    print(f"  {'Char':<6} {'h mean':>8} {'h std':>8} {'h min':>8} {'h max':>8}")
    print(f"  {'─'*6} {'─'*8} {'─'*8} {'─'*8} {'─'*8}")

    for ch in chars:
        # One-hot encode
        x = torch.zeros(1, len(vocab))
        x[0, c2i[ch]] = 1.0

        h = cell(x, h)

        print(f"  '{ch}'    "
              f"{h.mean().item():>8.4f} "
              f"{h.std().item():>8.4f} "
              f"{h.min().item():>8.4f} "
              f"{h.max().item():>8.4f}")

    print(f"\n  → Hidden state evolves with each character,")
    print(f"    carrying information about the past sequence.\n")


# ══════════════════════════════════════════════
#  7. RUN EVERYTHING
# ══════════════════════════════════════════════

if __name__ == "__main__":
    print("=" * 60)
    print("  Vanilla RNN Cell — From Scratch")
    print("=" * 60 + "\n")

    step_by_step()
    verify()
    sequence_classification()
    char_demo()

    print("""══════════════════════════════════════════════════════════════
  VANILLA RNN — HOW IT WORKS

  The core equation (one timestep):

      h_t = tanh( W_ih · x_t  +  W_hh · h_{t-1}  +  bias )
            ────   ──────────     ─────────────
            squash  new input     memory from past

  Unrolled through time:

      x₀ → [RNN Cell] → h₀
                ↓
      x₁ → [RNN Cell] → h₁     (same weights, reused!)
                ↓
      x₂ → [RNN Cell] → h₂
                ↓
             output

  Key points:
  • Same weights W_ih, W_hh are shared across ALL timesteps
  • h_t carries compressed memory of everything seen so far
  • tanh keeps values in [-1, 1] (but causes vanishing gradients)
  • For long sequences → use LSTM or GRU instead
══════════════════════════════════════════════════════════════
""")